# Rede Neural Generativa: DCGAN para gerar dígitos manuscritos

**Disciplina:** Processamento Digital de Sinais e Imagens (PDSI)

## Contexto

Uma **GAN** (*Generative Adversarial Network*, Goodfellow et al., 2014) é formada por duas redes que competem entre si:

- **Gerador (G):** recebe um vetor de ruído aleatório **z** e tenta produzir uma imagem que pareça real.
- **Discriminador (D):** recebe uma imagem e tenta dizer se ela é **real** (da base de dados) ou **falsa** (feita pelo gerador).

É como um falsificador (G) e um perito (D): à medida que o perito melhora, o falsificador é forçado a produzir falsificações cada vez mais convincentes. Formalmente, as duas redes disputam um jogo *minimax*:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{dados}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

Vamos usar uma **DCGAN** (*Deep Convolutional GAN*, Radford et al., 2015), em que G e D são redes convolucionais, para gerar dígitos parecidos com os do **MNIST**.

> 💡 No Colab, ative a GPU em *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Com GPU, o treino leva poucos minutos.

## Parte 1 – Bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from torch.utils.data import DataLoader

torch.manual_seed(42)                                         # resultados reprodutíveis
device = "cuda" if torch.cuda.is_available() else "cpu"      # usa a GPU, se disponível
print("Dispositivo:", device)

## Parte 2 – Base de dados (MNIST)

As imagens são normalizadas para **[−1, 1]**, o mesmo intervalo da função `tanh` usada na saída do gerador.

In [ ]:
transformacao = transforms.Compose([
    transforms.ToTensor(),                                    # [0, 255] -> [0, 1]
    transforms.Normalize([0.5], [0.5]),                       # [0, 1] -> [-1, 1]
])
mnist = datasets.MNIST(root="dados", train=True, download=True, transform=transformacao)
loader = DataLoader(mnist, batch_size=128, shuffle=True, drop_last=True)
print("Imagens de treino:", len(mnist))

def mostrar_grade(imagens, titulo, linhas=8):
    grade = make_grid(imagens[:linhas * linhas].cpu(), nrow=linhas, normalize=True, value_range=(-1, 1))  # monta um mosaico
    plt.figure(figsize=(6, 6)); plt.imshow(grade.permute(1, 2, 0)); plt.title(titulo); plt.axis("off"); plt.show()

reais, _ = next(iter(loader))
mostrar_grade(reais, "Imagens reais do MNIST")

## Parte 3 – As duas redes

### Gerador
Transforma um vetor de ruído **z** (100 valores) em uma imagem 28×28. As **convoluções transpostas** (`ConvTranspose2d`) fazem o caminho inverso de uma convolução comum: **aumentam** a resolução (7×7 → 14×14 → 28×28).

In [ ]:
Z_DIM = 100                                                   # tamanho do vetor de ruído (espaço latente)

class Gerador(nn.Module):
    def __init__(self):
        super().__init__()
        self.projecao = nn.Sequential(
            nn.Linear(Z_DIM, 128 * 7 * 7), nn.BatchNorm1d(128 * 7 * 7), nn.ReLU(True))  # z -> 128 mapas de 7x7
        self.convs = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),  # 7x7 -> 14x14
            nn.ConvTranspose2d(64, 1, 4, stride=2, padding=1), nn.Tanh())                           # 14x14 -> 28x28, saída em [-1, 1]

    def forward(self, z):
        x = self.projecao(z).view(-1, 128, 7, 7)              # vetor -> tensor (lote, canais, 7, 7)
        return self.convs(x)

### Discriminador
Uma CNN de classificação binária comum: recebe uma imagem 28×28 e devolve **uma pontuação** (*logit*) que indica se ela é real. Usamos `LeakyReLU`, que mantém um pequeno gradiente para valores negativos e estabiliza o treino.

In [ ]:
class Discriminador(nn.Module):
    def __init__(self):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Conv2d(1, 64, 4, stride=2, padding=1), nn.LeakyReLU(0.2, True),                         # 28x28 -> 14x14
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),  # 14x14 -> 7x7
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1))                                                                 # 1 logit: real ou falsa

    def forward(self, x):
        return self.rede(x)

In [ ]:
def iniciar_pesos(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight, 0.0, 0.02)                  # inicialização recomendada no artigo da DCGAN
    elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)

G = Gerador().to(device); G.apply(iniciar_pesos)
D = Discriminador().to(device); D.apply(iniciar_pesos)
print(f"Parâmetros do gerador: {sum(p.numel() for p in G.parameters()):,}")
print(f"Parâmetros do discriminador: {sum(p.numel() for p in D.parameters()):,}")

# Antes do treino, o gerador só produz ruído
with torch.no_grad():
    mostrar_grade(G.eval()(torch.randn(64, Z_DIM, device=device)), "Saída do gerador ANTES do treino")

## Parte 4 – Função de perda e otimizadores

Usamos a **entropia cruzada binária** (`BCEWithLogitsLoss`, que já aplica a sigmoide) e o otimizador **Adam** com os hiperparâmetros sugeridos no artigo da DCGAN.

In [ ]:
criterio = nn.BCEWithLogitsLoss()
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

ruido_fixo = torch.randn(64, Z_DIM, device=device)           # mesmo ruído em todas as épocas, para ver a evolução
EPOCAS = 10

## Parte 5 – Treinamento adversarial

A cada lote, há dois passos:

1. **Treina D:** mostra imagens reais (rótulo 1) e falsas geradas por G (rótulo 0), e D aprende a separá-las.
2. **Treina G:** gera imagens falsas e calcula a perda **como se fossem reais** (rótulo 1). G aprende a "enganar" D.

In [ ]:
hist = {"perda_D": [], "perda_G": [], "D_real": [], "D_falsa": []}
amostras_por_epoca = []

for epoca in range(EPOCAS):
    G.train(); D.train()
    for reais, _ in loader:
        reais = reais.to(device)
        n = reais.size(0)
        um, zero = torch.ones(n, 1, device=device), torch.zeros(n, 1, device=device)

        # ---------- 1. Treina o discriminador ----------
        falsas = G(torch.randn(n, Z_DIM, device=device))
        saida_real, saida_falsa = D(reais), D(falsas.detach())   # detach: não atualiza G neste passo
        perda_D = criterio(saida_real, um) + criterio(saida_falsa, zero)
        opt_D.zero_grad(); perda_D.backward(); opt_D.step()

        # ---------- 2. Treina o gerador ----------
        perda_G = criterio(D(falsas), um)                      # G quer que D classifique as falsas como reais
        opt_G.zero_grad(); perda_G.backward(); opt_G.step()

        hist["perda_D"].append(perda_D.item()); hist["perda_G"].append(perda_G.item())
        hist["D_real"].append(torch.sigmoid(saida_real).mean().item())    # probabilidade média que D dá às reais
        hist["D_falsa"].append(torch.sigmoid(saida_falsa).mean().item())  # probabilidade média que D dá às falsas

    G.eval()
    with torch.no_grad():
        amostras_por_epoca.append(G(ruido_fixo).cpu())         # guarda as imagens geradas com o ruído fixo
    print(f"Época {epoca + 1:2d}/{EPOCAS} | perda D: {np.mean(hist['perda_D'][-len(loader):]):.3f} | "
          f"perda G: {np.mean(hist['perda_G'][-len(loader):]):.3f} | "
          f"D(real): {np.mean(hist['D_real'][-len(loader):]):.2f} | D(falsa): {np.mean(hist['D_falsa'][-len(loader):]):.2f}")

## Parte 6 – Analisando o treino

Diferente de uma rede comum, a perda de uma GAN **não precisa cair até zero**: as duas redes competem, e o ideal é um equilíbrio. Se D ficar perfeito (D(real) → 1 e D(falsa) → 0), G para de aprender.

In [ ]:
suavizar = lambda v, k=50: np.convolve(v, np.ones(k) / k, mode="valid")   # média móvel para limpar o gráfico

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(suavizar(hist["perda_D"]), label="Discriminador"); axes[0].plot(suavizar(hist["perda_G"]), label="Gerador")
axes[0].set_title("Perdas durante o treino"); axes[0].set_xlabel("Iteração"); axes[0].legend()
axes[1].plot(suavizar(hist["D_real"]), label="D(x real)"); axes[1].plot(suavizar(hist["D_falsa"]), label="D(G(z))")
axes[1].axhline(0.5, ls="--", c="gray"); axes[1].set_ylim(0, 1)
axes[1].set_title("Probabilidade de 'real' atribuída por D"); axes[1].set_xlabel("Iteração"); axes[1].legend()
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Evolução das imagens geradas (mesmo ruído em todas as épocas)

In [ ]:
epocas_mostradas = sorted(set([0, 1, EPOCAS // 2, EPOCAS - 1]))  # algumas épocas para comparar
plt.figure(figsize=(4 * len(epocas_mostradas), 4))
for k, e in enumerate(epocas_mostradas):
    grade = make_grid(amostras_por_epoca[e][:36], nrow=6, normalize=True, value_range=(-1, 1))
    plt.subplot(1, len(epocas_mostradas), k + 1); plt.imshow(grade.permute(1, 2, 0)); plt.title(f"Época {e + 1}"); plt.axis("off")
plt.tight_layout(); plt.show()

## Parte 7 – Gerando novos dígitos

In [ ]:
G.eval()
with torch.no_grad():
    novas = G(torch.randn(64, Z_DIM, device=device))          # ruído novo -> imagens nunca vistas
mostrar_grade(novas, "Dígitos gerados pela DCGAN")

### Interpolação no espaço latente

Caminhando em linha reta entre dois vetores de ruído **z₁** e **z₂**, o gerador produz uma transição suave entre dois dígitos. Isso mostra que a rede aprendeu um espaço latente **contínuo**, e não apenas "decorou" imagens.

In [ ]:
with torch.no_grad():
    linhas = []
    for _ in range(4):                                         # 4 pares de pontos aleatórios
        z1, z2 = torch.randn(1, Z_DIM, device=device), torch.randn(1, Z_DIM, device=device)
        alphas = torch.linspace(0, 1, 10, device=device).view(-1, 1)
        z = (1 - alphas) * z1 + alphas * z2                    # 10 pontos entre z1 e z2
        linhas.append(G(z).cpu())
grade = make_grid(torch.cat(linhas), nrow=10, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(12, 5)); plt.imshow(grade.permute(1, 2, 0)); plt.title("Interpolação entre z₁ (esquerda) e z₂ (direita)"); plt.axis("off"); plt.show()

## Parte 8 – Salvando o gerador

In [ ]:
torch.save(G.state_dict(), "gerador_dcgan_mnist.pth")       # para gerar imagens depois, basta o gerador
print("Gerador salvo!")

## Perguntas para discussão

1. Por que o discriminador é treinado com `falsas.detach()`, mas o gerador não?
2. Observe o gráfico de D(x real) e D(G(z)). O que significaria se as duas curvas fossem para 0,5?
3. A perda do gerador subiu em algum momento, mesmo com as imagens melhorando? Por que a perda de uma GAN não mede diretamente a qualidade das imagens?
4. Troque `datasets.MNIST` por `datasets.FashionMNIST` e treine de novo. O que a GAN passa a gerar?
5. O que é o *mode collapse* (colapso de modo)? Você vê sinais dele nas imagens geradas (ex.: muitos dígitos iguais)?

## Referências

- GOODFELLOW, I. et al. *Generative Adversarial Nets*. In: Advances in Neural Information Processing Systems (NeurIPS), 2014. Disponível em: https://arxiv.org/abs/1406.2661
- RADFORD, A.; METZ, L.; CHINTALA, S. *Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks*. In: ICLR, 2016. Disponível em: https://arxiv.org/abs/1511.06434
- Tutorial oficial de DCGAN do PyTorch: https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html
